# Configuration Dataclasses

This notebook defines the configuration dataclasses used throughout the GPT project.
There are three configs:

- **GPTConfig** — model architecture hyperparameters
- **TrainConfig** — pretraining pipeline settings
- **SFTConfig** — supervised fine-tuning settings

All hyperparameters flow through these dataclasses, enabling reproducibility
and checkpoint logging.

In [ ]:
from dataclasses import dataclass

## GPTConfig

Defines the model architecture. Key fields:

| Field | Default | Description |
|---|---|---|
| `vocab_size` | 8000 | Number of tokens in the BPE vocabulary |
| `d_model` | 256 | Embedding / hidden dimension |
| `num_heads` | 8 | Number of attention heads (d_model must be divisible by this) |
| `num_layers` | 6 | Number of stacked transformer blocks |
| `max_seq_len` | 512 | Maximum input sequence length |
| `dropout_rate` | 0.1 | Dropout probability applied after attention and MLP |

The `validate()` method checks that all values are positive, dropout is in [0, 1],
and `d_model` is evenly divisible by `num_heads`.

In [ ]:
@dataclass
class GPTConfig:
    """Configuration for the GPT model architecture."""

    vocab_size: int = 8000
    d_model: int = 256
    num_heads: int = 8
    num_layers: int = 6
    max_seq_len: int = 512
    dropout_rate: float = 0.1

    def validate(self) -> None:
        """Validate configuration values.

        Raises:
            ValueError: If d_model is not divisible by num_heads,
                        if any numeric value is non-positive,
                        or if dropout_rate is outside [0, 1].
        """
        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be positive, got {self.vocab_size}")
        if self.d_model <= 0:
            raise ValueError(f"d_model must be positive, got {self.d_model}")
        if self.num_heads <= 0:
            raise ValueError(f"num_heads must be positive, got {self.num_heads}")
        if self.num_layers <= 0:
            raise ValueError(f"num_layers must be positive, got {self.num_layers}")
        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be positive, got {self.max_seq_len}")
        if self.dropout_rate < 0 or self.dropout_rate > 1:
            raise ValueError(
                f"dropout_rate must be between 0 and 1, got {self.dropout_rate}"
            )
        if self.d_model % self.num_heads != 0:
            raise ValueError(
                f"d_model ({self.d_model}) must be divisible by num_heads ({self.num_heads})"
            )

## TrainConfig

Controls the pretraining pipeline — optimizer, schedule, checkpointing, and early stopping.

| Field | Default | Description |
|---|---|---|
| `learning_rate` | 3e-4 | Peak learning rate for AdamW |
| `weight_decay` | 0.01 | L2 regularization strength |
| `beta1` / `beta2` | 0.9 / 0.999 | Adam momentum parameters |
| `batch_size` | 32 | Sequences per training step |
| `max_steps` | 10000 | Total training steps |
| `warmup_steps` | 500 | Linear LR warmup duration |
| `log_interval` | 100 | Steps between loss logging |
| `eval_interval` | 500 | Steps between validation evaluations |
| `save_interval` | 1000 | Steps between checkpoint saves |
| `patience` | 3 | Consecutive val-loss increases before overfitting warning |
| `seed` | 42 | Random seed for reproducibility |
| `train_split` | 0.9 | Fraction of corpus used for training (rest is validation) |

In [ ]:
@dataclass
class TrainConfig:
    """Configuration for the pretraining pipeline."""

    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    beta1: float = 0.9
    beta2: float = 0.999
    batch_size: int = 32
    max_steps: int = 10000
    warmup_steps: int = 500
    log_interval: int = 100
    eval_interval: int = 500
    save_interval: int = 1000
    patience: int = 3
    seed: int = 42
    train_split: float = 0.9

## SFTConfig

Controls supervised fine-tuning on instruction-response pairs.
Uses a lower learning rate than pretraining to avoid catastrophic forgetting.

| Field | Default | Description |
|---|---|---|
| `learning_rate` | 1e-5 | Lower LR to preserve pretrained knowledge |
| `batch_size` | 16 | Sequences per fine-tuning step |
| `max_steps` | 2000 | Total fine-tuning steps |
| `log_interval` | 50 | Steps between loss logging |
| `eval_interval` | 200 | Steps between validation evaluations |
| `save_interval` | 500 | Steps between checkpoint saves |
| `separator_tokens` | `\n### ` | Delimiter between instruction and output in formatted examples |
| `seed` | 42 | Random seed for reproducibility |

In [ ]:
@dataclass
class SFTConfig:
    """Configuration for the supervised fine-tuning pipeline."""

    learning_rate: float = 1e-5
    batch_size: int = 16
    max_steps: int = 2000
    log_interval: int = 50
    eval_interval: int = 200
    save_interval: int = 500
    separator_tokens: str = "\n### "
    seed: int = 42

## Quick Demo

Create instances with defaults and verify validation works.

In [ ]:
gpt_cfg = GPTConfig()
gpt_cfg.validate()
print("GPTConfig:", gpt_cfg)

train_cfg = TrainConfig()
print("TrainConfig:", train_cfg)

sft_cfg = SFTConfig()
print("SFTConfig:", sft_cfg)

In [ ]:
# Validation catches invalid configs
try:
    bad = GPTConfig(d_model=100, num_heads=3)
    bad.validate()
except ValueError as e:
    print("Caught expected error:", e)